# Lab 3: Architectures in Practice

Adapted from 6.7960 Fall 2025 HW2.

A CNN and a Vision Transformer begin with different assumptions about
images. The source homework trains its CNN on CIFAR-100 and its ViT on
CIFAR-10, so those results cannot be compared directly. This lab puts
both compact, fully implemented models on the same seeded CIFAR-10
split, augmentation, epoch budget, and optimizer.

You will audit shapes and parameter counts, train matched baselines,
vary one architecture control at a time, and inspect class-token
attention. Measured accuracy is evidence for reflection, not a graded
value.


## Runtime and reproducibility

QUICK mode is the free-Colab default: 8,000 training examples and
three epochs. Set the environment variable DL_LAB_MODE to full before
running this cell for all CIFAR-10 examples and ten epochs.


In [ ]:
import math, os, random, time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
MODE = os.environ.get("DL_LAB_MODE", "quick").lower()
if MODE not in {"quick", "full"}:
    raise ValueError("DL_LAB_MODE must be quick or full")
QUICK_MODE = MODE == "quick"

def seed_all(seed=SEED):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except AttributeError:
        pass

seed_all()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SETTINGS = {
    "quick": dict(train_n=8000, val_n=2000, epochs=3, batch=128),
    "full": dict(train_n=40000, val_n=10000, epochs=10, batch=256),
}[MODE]
print("mode:", MODE, "device:", DEVICE, "settings:", SETTINGS)


## Matched CIFAR-10 data

A seeded 40,000/10,000 split is drawn from CIFAR-10's training
partition. QUICK mode uses prefixes of those same two pools. Both
models receive identical indexed examples, and only the training
loader applies random crop and horizontal flip. The official
CIFAR-10 test partition remains reserved and is not loaded during
architecture tuning.


In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)
])
train_aug_base = datasets.CIFAR10(
    "data", train=True, download=True, transform=train_tf
)
train_eval_base = datasets.CIFAR10(
    "data", train=True, download=False, transform=eval_tf
)
g = torch.Generator().manual_seed(SEED)
split_order = torch.randperm(len(train_aug_base), generator=g)
train_pool, val_pool = split_order[:40000], split_order[40000:]
train_ids = train_pool[:SETTINGS["train_n"]]
val_ids = val_pool[:SETTINGS["val_n"]]
train_data = Subset(train_aug_base, train_ids.tolist())
val_data = Subset(train_eval_base, val_ids.tolist())

def loader(data, train=False):
    return DataLoader(
        data, batch_size=SETTINGS["batch"], shuffle=train,
        num_workers=0, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(SEED),
    )

train_loader, val_loader = loader(train_data, True), loader(val_data)
x, y = next(iter(val_loader))
print(
    "seeded train/validation:", len(train_data), len(val_data),
    "official test: reserved and not loaded",
)
print(tuple(x.shape), tuple(y.shape))


## Prefilled CNN and ViT

CNN depth and first-kernel width are exposed controls. The compact ViT
exposes patch size, head count, and layer count. Its embedding width
is 96 so the default three heads divide evenly.


In [ ]:
def nparams(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

class SmallCNN(nn.Module):
    def __init__(self, depth=4, first_kernel=5, classes=10):
        super().__init__()
        if depth not in {3, 4, 5} or first_kernel not in {3, 5}:
            raise ValueError("depth must be 3-5 and first_kernel 3 or 5")
        layers, cin = [], 3
        for i in range(depth):
            cout = min(32 * 2 ** i, 192)
            k, stride = (first_kernel if i == 0 else 3), (1 if i == 0 else 2)
            layers += [
                nn.Conv2d(cin, cout, k, stride, k // 2, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            ]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, classes)

    def forward(self, x):
        x = self.features(x)
        return self.head(F.adaptive_avg_pool2d(x, 1).flatten(1))

class PatchEmbed(nn.Module):
    def __init__(self, patch=4, dim=96):
        super().__init__()
        if 32 % patch:
            raise ValueError("patch must divide 32")
        self.num_patches = (32 // patch) ** 2
        self.proj = nn.Conv2d(3, dim, patch, patch)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Block(nn.Module):
    def __init__(self, dim=96, heads=3):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim)
        )

    def forward(self, x, return_attention=False):
        z = self.n1(x)
        a, weights = self.attn(
            z, z, z, need_weights=return_attention,
            average_attn_weights=False,
        )
        x = x + a
        return x + self.mlp(self.n2(x)), weights

class CompactViT(nn.Module):
    def __init__(self, patch=4, dim=96, heads=3, layers=3, classes=10):
        super().__init__()
        if dim % heads:
            raise ValueError("dim must divide evenly across heads")
        self.patch_embed = PatchEmbed(patch, dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, self.token_count, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(layers)])
        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, classes)
        nn.init.normal_(self.cls, std=.02); nn.init.normal_(self.pos, std=.02)

    @property
    def token_count(self):
        return self.patch_embed.num_patches + 1

    def forward(self, images, return_attention=False):
        patches = self.patch_embed(images)
        cls = self.cls.expand(images.shape[0], -1, -1)
        x = torch.cat([cls, patches], 1)
        x = x + self.pos[:, :x.shape[1]]
        stack = []
        for block in self.blocks:
            x, weights = block(x, return_attention)
            if return_attention:
                stack.append(weights)
        logits = self.head(self.norm(x)[:, 0])
        attention = torch.stack(stack, 1) if return_attention else None
        return logits, attention


## Architecture audit and deterministic token diagnostic

The source configuration makes 8 by 8 image patches. The Transformer
receives those 64 patch tokens plus one class token: exactly 65.


In [ ]:
@torch.no_grad()
def audit(name, model):
    model.eval()
    out = model(torch.zeros(2, 3, 32, 32))
    logits = out[0] if isinstance(out, tuple) else out
    row = dict(name=name, parameters=nparams(model), output=tuple(logits.shape))
    if isinstance(model, CompactViT):
        row["tokens"] = model.token_count
    print(row)
    return row

CNN_BASE = dict(depth=4, first_kernel=5)
VIT_BASE = dict(patch=4, dim=96, heads=3, layers=3)
for d in (3, 4, 5): audit("cnn_depth_" + str(d), SmallCNN(d, 5))
audit("cnn_kernel_3", SmallCNN(4, 3))
for name, delta in {
    "vit_default": {}, "vit_patch_8": {"patch": 8},
    "vit_heads_1": {"heads": 1}, "vit_layers_2": {"layers": 2},
    "vit_layers_6": {"layers": 6},
}.items():
    audit(name, CompactViT(**{**VIT_BASE, **delta}))

TOKEN_DIAGNOSTIC = CompactViT(**VIT_BASE).token_count
assert TOKEN_DIAGNOSTIC == 65
print("DETERMINISTIC TOKEN DIAGNOSTIC =", TOKEN_DIAGNOSTIC)


## Matched training loop

Both families use AdamW and cosine decay. All rows printed below are
measurements from this runtime; this notebook ships no fake outputs.


In [ ]:
def get_logits(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else out

@torch.no_grad()
def evaluate(model):
    model.eval(); correct = count = loss_sum = 0
    for x, y in val_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = get_logits(model, x)
        loss_sum += F.cross_entropy(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item(); count += len(y)
    return dict(loss=loss_sum / count, accuracy=correct / count)

def fit(name, model, lr=5e-4):
    seed_all(); model = model.to(DEVICE)
    matched_train_loader = loader(train_data, True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, SETTINGS["epochs"])
    history = []
    for epoch in range(1, SETTINGS["epochs"] + 1):
        model.train(); correct = count = loss_sum = 0
        started = time.perf_counter()
        for x, y in matched_train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = get_logits(model, x)
            loss = F.cross_entropy(logits, y); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item(); count += len(y)
        sched.step(); val = evaluate(model)
        row = dict(
            epoch=epoch, train_loss=loss_sum / count,
            train_accuracy=correct / count, val_accuracy=val["accuracy"],
            seconds=time.perf_counter() - started,
        )
        history.append(row); print(name, row)
    return dict(model=model, history=history, parameters=nparams(model))

RUNS = {
    "cnn_baseline": fit("cnn_baseline", SmallCNN(**CNN_BASE)),
    "vit_baseline": fit("vit_baseline", CompactViT(**VIT_BASE)),
}


## Controlled tuning and report

Choose one candidate from each family. The audit is cheap; training
every cross-product is not. Keep all other settings fixed and set
RUN_TUNING to True.


In [ ]:
CNN_OPTIONS = {
    "cnn_depth_3": {**CNN_BASE, "depth": 3},
    "cnn_depth_5": {**CNN_BASE, "depth": 5},
    "cnn_kernel_3": {**CNN_BASE, "first_kernel": 3},
}
VIT_OPTIONS = {
    "vit_patch_8": {**VIT_BASE, "patch": 8},
    "vit_heads_1": {**VIT_BASE, "heads": 1},
    "vit_layers_2": {**VIT_BASE, "layers": 2},
    "vit_layers_6": {**VIT_BASE, "layers": 6},
}
CHOSEN_CNN, CHOSEN_VIT, RUN_TUNING = "cnn_depth_3", "vit_patch_8", False
if RUN_TUNING:
    RUNS[CHOSEN_CNN] = fit(CHOSEN_CNN, SmallCNN(**CNN_OPTIONS[CHOSEN_CNN]))
    RUNS[CHOSEN_VIT] = fit(CHOSEN_VIT, CompactViT(**VIT_OPTIONS[CHOSEN_VIT]))
else:
    print("Choose configurations and set RUN_TUNING=True.")

for name, run in RUNS.items():
    final = run["history"][-1]
    print(dict(
        name=name, parameters=run["parameters"],
        final_val_accuracy=final["val_accuracy"],
        median_epoch_seconds=float(np.median([r["seconds"] for r in run["history"]])),
    ))

for name, run in RUNS.items():
    plt.plot(
        [r["epoch"] for r in run["history"]],
        [r["val_accuracy"] for r in run["history"]],
        marker="o", label=name,
    )
plt.xlabel("epoch"); plt.ylabel("validation accuracy")
plt.title("Matched CIFAR-10 runs"); plt.legend(); plt.grid(alpha=.25)
plt.show()


## Class-token attention

The hook averages the class token's patch attention across all layers
and heads. Attention is a useful diagnostic, not a causal explanation.


In [ ]:
def undo_normalize(image):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

@torch.no_grad()
def show_attention(model, index=0):
    model.eval(); images, labels = next(iter(val_loader))
    logits, weights = model(images[index:index+1].to(DEVICE), True)
    patch_weights = weights[0, :, :, 0, 1:].mean((0, 1)).cpu()
    side = math.isqrt(patch_weights.numel())
    assert side * side == patch_weights.numel()
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(undo_normalize(images[index]).permute(1, 2, 0))
    axes[0].set_title(
        "true=" + str(labels[index].item()) +
        ", predicted=" + str(logits.argmax(1).item())
    )
    axes[1].imshow(patch_weights.reshape(side, side), cmap="magma")
    axes[1].set_title("mean class-token attention")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()
    return patch_weights

attention_map = show_attention(RUNS["vit_baseline"]["model"])


## What to record

Record what changed when you altered CNN locality or ViT patch size.
The edX submission grades the hand-checkable token count and robust
architecture tradeoffs. It never asks every learner to reproduce one
noisy validation score.
